# Clase 3 · Laboratorio integral de clasificación multiclase
## De los logits hasta softmax, NLL, entropía cruzada, accuracy y divergencia KL

**Referencia directa:** presentación *Clase 3 — Clasificación Multiclase: explicación detallada* (56 diapositivas), asociada al capítulo 5 de *Understanding Deep Learning*, Simon J. D. Prince.

**Caso práctico:** clasificación de fotografías de infraestructuras como **puente**, **edificio** o **carretera**. Todos los valores son *sintéticos* con propósito didáctico; no provienen de imágenes reales ni de una red entrenada.

### Objetivos

1. Identificar entradas, capas, pesos, sesgos y **logits** de una red con tres clases.
2. Calcular *paso por paso* exponentes, denominador, **softmax** y `argmax`.
3. Representar las etiquetas verdaderas como índices y vectores **one-hot**.
4. Deducir y calcular **NLL**, **Cross-Entropy (CE)** y sus valores por lote.
5. Comparar **accuracy**, pérdida media, entropía $H(q)$ y divergencia $D_{KL}(q\|p)$.
6. Explicar **invarianza de softmax**, **estabilidad numérica** y la diferencia entre multiclase y multietiqueta.
7. Modificar logits y observar cómo cambia cada indicador.

**Duración sugerida:** 85–105 minutos. **Requisitos:** Python 3, NumPy y Matplotlib. No requiere PyTorch, TensorFlow ni Internet.

> **Convenciones:** `np.log` es el logaritmo *natural*, `np.exp` la exponencial base $e$, los índices de clases en Python comienzan en 0, y todas las pérdidas están expresadas en **nats** (logaritmos naturales). Las probabilidades mostradas se redondean, pero los cálculos usan precisión completa.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
CLASES = np.array(['Puente', 'Edificio', 'Carretera'])
print('Clases:', dict(enumerate(CLASES)))
print('NumPy:', np.__version__)

## 1. ¿Cuál es el problema y qué debe producir la red? (diapositivas 2–6)

La entrada $x$ sería una imagen. Una red neuronal transforma la imagen en características internas $h$ y finalmente calcula **una puntuación por clase**:

$$\underbrace{x}_{\text{fotografía}}\ \xrightarrow{f_\phi}\ \underbrace{h}_{\text{características}}\ \xrightarrow{W,b}\ \underbrace{z}_{\text{K logits}}\ \xrightarrow{\operatorname{softmax}}\ \underbrace{p}_{\text{K probabilidades}}.$$

El vector final debe representar clases **mutuamente excluyentes**: si la fotografía ha sido anotada únicamente como puente, su etiqueta no es también carretera. Con $K=3$, la red produce tres logits y, posteriormente, tres probabilidades cuya suma es uno.

**Contraste importante:** si una imagen pudiera tener simultáneamente `lluvia`, `tráfico` y `peatones`, sería un caso **multietiqueta**. En ese problema sería habitual aplicar una sigmoide y BCE a cada etiqueta, no una única softmax que fuerce competencia entre ellas.

## 2. Logits: antes de hablar de probabilidades (diapositivas 7–9)

Una última capa afín calcula:

$$\boxed{z=Wh+b}$$

- $h\in\mathbb R^D$: vector de activaciones procedente de la capa oculta previa.
- $W\in\mathbb R^{K\times D}$: cada **fila** corresponde a los pesos de una clase.
- $b\in\mathbb R^K$: sesgo por clase.
- $z\in\mathbb R^K$: **logits**; números reales sin normalizar, **no porcentajes**.

**Ejemplo construido para obtener exactamente** $z=[2,1,0]$: tomemos $h=[1,2]$ y una salida lineal de tres neuronas.

In [ ]:
h = np.array([1.0, 2.0])
W = np.array([[2.0, 0.0],
              [0.0, 0.5],
              [1.0, -0.5]])
b = np.zeros(3)
z = W @ h + b
print('h =', h, 'forma:', h.shape)
print('W =\n', W, 'forma:', W.shape)
print('b =', b, 'forma:', b.shape)
for k, name in enumerate(CLASES):
    print(f'  z[{k}] ({name}) = {W[k]} · {h} + {b[k]} = {z[k]:.3f}')
print('Logits finales:', z)
assert np.allclose(z, [2, 1, 0])

### Interpretación de pesos y sesgos

Un aumento del sesgo $b_k$ desplaza el logit de la clase $k$; cambiar $W_{kj}$ modifica cuánto contribuye la característica $h_j$ a esa clase. **No debe interpretarse** que $z_k=2$ significa «200%»: el resultado depende también de las otras clases y todavía no está normalizado.

Para decidir qué clase obtiene la puntuación mayor usamos $\arg\max_k z_k$, pero para evaluar CE necesitamos las **probabilidades completas**.

In [ ]:
ganadora_logits = int(np.argmax(z))
print('Clase con mayor logit:', CLASES[ganadora_logits])
print('¿Suman 1 los logits?', np.isclose(z.sum(), 1.0))

## 3. Softmax: deducción operacional completa (diapositivas 10–17)

Para cada clase $k\in\{1,\ldots,K\}$,

$$\boxed{p_k=\operatorname{softmax}(z)_k=\frac{e^{z_k}}{\displaystyle\sum_{j=1}^{K}e^{z_j}}}$$

**Lectura detallada:**

1. **Numerador** $e^{z_k}$: convierte la puntuación $z_k$ en un valor estrictamente positivo.
2. **Denominador** $S=\sum_j e^{z_j}$: suma las contribuciones de **todas** las clases; es el mismo para cada $p_k$.
3. **Normalización** $p_k=e^{z_k}/S$: da proporciones positivas que suman $1$.
4. **Interpretación** $p_k$: preferencia probabilística del modelo por la clase $k$ **bajo su representación aprendida**. No garantiza calibración estadística.

Para $z=[2,1,0]$ calcularemos los cinco pasos sin ocultar operaciones.

In [ ]:
z_demo = np.array([2.0, 1.0, 0.0])
exps = np.exp(z_demo)
S = exps.sum()
p_manual = exps / S
for k, name in enumerate(CLASES):
    print(f'{name:10} | z = {z_demo[k]:.0f} | e^z = {exps[k]:.6f} | p = {exps[k]:.6f}/{S:.6f} = {p_manual[k]:.6f} ({100*p_manual[k]:.2f}%)')
print(f'Denominador S = {S:.6f}')
print(f'Comprobación Σp = {p_manual.sum():.12f}')
print('Clase predicha:', CLASES[np.argmax(p_manual)])
assert np.allclose(p_manual, [0.665240956, 0.244728471, 0.090030573])

### Gráfico 1: logits frente a probabilidades

Las barras de **logits** pueden ser negativas y no suman uno; las barras de **probabilidades softmax** expresan una distribución normalizada. Compararemos ambos valores sin confundir sus escalas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(CLASES, z_demo)
axes[0].set_title('Logits (puntuaciones)')
axes[0].set_ylabel('Valor sin normalizar')
axes[1].bar(CLASES, p_manual)
axes[1].set_title('Probabilidades después de softmax')
axes[1].set_ylabel('Probabilidad')
axes[1].set_ylim(0, 1)
for i, val in enumerate(p_manual): axes[1].text(i, val+0.025, f'{val:.3f}', ha='center')
plt.tight_layout(); plt.show()

## 4. ¿Qué cambia al modificar los logits? (diapositivas 18–20)

La softmax no trabaja con cada clase de forma independiente: aumentar **un solo logit** aumenta su probabilidad y **reduce las restantes**, aunque los otros logits no cambien.

**Invarianza a una traslación común:** para cualquier constante $c$,

$$\frac{e^{z_k+c}}{\sum_j e^{z_j+c}}=\frac{e^c e^{z_k}}{e^c\sum_j e^{z_j}}=\frac{e^{z_k}}{\sum_j e^{z_j}}.$$

Por tanto, sumar la misma constante a **todos** los logits no cambia ninguna probabilidad. Es diferente de cambiar solo uno.

In [ ]:
def softmax_estable(logits, axis=-1):
    """Softmax numéricamente estable para un vector o una matriz de filas."""
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)

casos = {
    'Original': [2, 1, 0],
    'Sube solo B': [2, 3, 0],
    'Suma +100 a todos': [102, 101, 100],
    'Sube solo A': [3, 1, 0],
}
for etiqueta, logits in casos.items():
    print(f'{etiqueta:21} z={logits} -> p={np.round(softmax_estable(logits), 6)}')
assert np.allclose(softmax_estable([2,1,0]), softmax_estable([102,101,100]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
labels = list(casos)
pos = np.arange(3)
width = 0.18
for i, (name, logits) in enumerate(casos.items()):
    ax.bar(pos + (i-1.5)*width, softmax_estable(logits), width=width, label=name)
ax.set_xticks(pos, CLASES)
ax.set_ylim(0, 1)
ax.set_ylabel('Probabilidad softmax')
ax.set_title('Competencia entre clases: cambio de logits')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Etiqueta verdadera e indicador one-hot (diapositivas 21–22)

El modelo no debe confundirse con la verdad observada:

- $p=[p_A,p_B,p_C]$: **probabilidades predichas**.
- $q=[q_A,q_B,q_C]$: **distribución objetivo**; con etiqueta one-hot contiene exactamente un $1$.

Si la clase real es **B = edificio** (índice 1 en Python), entonces $q=[0,1,0]$. La red puede seguir prediciendo `Puente`, y en tal caso estará equivocada.

$$q_k=\begin{cases}1,&k=y_{\mathrm{índice}}\\0,&k\ne y_{\mathrm{índice}}\end{cases}$$

In [ ]:
def one_hot(indice, K=3):
    q = np.zeros(K, dtype=float)
    q[indice] = 1.0
    return q

for clase_real in range(len(CLASES)):
    print(f'Clase real {CLASES[clase_real]:10} -> índice={clase_real}, one-hot={one_hot(clase_real).astype(int)}')
print('Predicción del modelo:', np.round(p_manual, 6))
print('Clase predicha:', CLASES[int(np.argmax(p_manual))])

## 6. Verosimilitud categórica y NLL (diapositivas 23–27)

Cuando solo una clase ocurre, la distribución categórica puede escribir su probabilidad mediante un producto:

$$P(y=k\mid x,\phi)=p_k,\qquad P(q\mid p)=\prod_{k=1}^K p_k^{q_k}.$$

Con $q$ one-hot, todos los exponentes son cero salvo uno; el producto se reduce a la probabilidad asignada a la **clase verdadera**.

La **pérdida negativa logarítmica** (NLL) por observación se define como:

$$\boxed{\ell_{\mathrm{NLL}}=-\ln p_y}$$

¿Por qué el signo negativo? $\ln(p_y)\le0$ si $0<p_y\le1$; su negativo es no negativo y **crece cuando el modelo asigna poca probabilidad a lo realmente ocurrido**.

**Ejemplo:** mantendremos $p=\mathrm{softmax}([2,1,0])$ y cambiaremos solamente la clase verdadera. Así veremos por qué la misma predicción puede ser buena o mala según la etiqueta observada.

In [ ]:
for y_indice, nombre in enumerate(CLASES):
    prob_verdadera = p_manual[y_indice]
    perdida = -np.log(prob_verdadera)
    print(f'Clase real {nombre:10}: p_y={prob_verdadera:.6f}; NLL=-ln({prob_verdadera:.6f})={perdida:.6f} nats')
print('Ejemplo extra: -ln(0.9)=', -np.log(0.9), '; -ln(0.01)=', -np.log(0.01))

## 7. Entropía cruzada (CE): fórmula y derivación (diapositivas 28–31)

La CE compara la distribución objetivo $q$ con la predicha $p$:

$$\boxed{H(q,p)=-\sum_{k=1}^{K}q_k\ln p_k}$$

**Explicación de cada símbolo:** $K$ es el número de clases; $q_k$ indica el peso de la clase $k$ en el objetivo; $p_k$ es la probabilidad predicha para esa clase; $\ln$ es el logaritmo natural; $-$ convierte la log-verosimilitud en pérdida a minimizar.

**Deducción desde la distribución categórica:**

$$P(q\mid p)=\prod_{k=1}^{K}p_k^{q_k}$$
$$-\ln P(q\mid p)=-\ln\Big(\prod_{k=1}^{K}p_k^{q_k}\Big)$$
$$\boxed{-\ln P(q\mid p)=-\sum_{k=1}^{K}q_k\ln p_k=H(q,p)}$$

Con **one-hot**, $q_j=1$ solo para la etiqueta verdadera y todos los otros sumandos se anulan: **CE = NLL = $-\ln p_y$**. Para objetivos suaves ($q$ no one-hot), participan varias clases.

In [ ]:
def cross_entropy(q, p):
    """CE para distribuciones en el simplex; requiere p positiva."""
    q = np.asarray(q, dtype=float)
    p = np.asarray(p, dtype=float)
    if q.shape != p.shape or not np.all(p > 0) or not np.allclose(p.sum(axis=-1), 1):
        raise ValueError('q y p deben tener formas compatibles y p ser una distribución positiva')
    if not np.all(q >= 0) or not np.allclose(q.sum(axis=-1), 1):
        raise ValueError('q debe ser una distribución válida')
    # El factor q=0 debe contribuir exactamente 0 sin producir 0*log(0).
    return float(-np.sum(np.where(q > 0, q * np.log(p), 0.0)))

q_B = one_hot(1)
contribuciones = -q_B * np.log(p_manual)
print('q (clase real Edificio) =', q_B)
print('p (modelo)             =', p_manual)
for k, nombre in enumerate(CLASES):
    print(f'  -q[{k}] ln(p[{k}]) = -{q_B[k]:.0f} ln({p_manual[k]:.6f}) = {contribuciones[k]:.6f}')
print('CE = suma de contribuciones =', cross_entropy(q_B, p_manual))
print('NLL = -ln(p_B)             =', -np.log(p_manual[1]))
assert np.isclose(cross_entropy(q_B, p_manual), -np.log(p_manual[1]))

### Tabla de pérdidas para la misma distribución softmax

Al cambiar únicamente la etiqueta verdadera, la pérdida cambia mucho aunque la red no produzca otros logits. La NLL para la clase **C** será la mayor porque la red le asignó menos probabilidad.

In [ ]:
losses_by_class = -np.log(p_manual)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(CLASES, losses_by_class)
ax.set_ylabel('NLL = −ln(p_y) (nats)')
ax.set_title('La misma predicción evaluada con tres etiquetas posibles')
for i, value in enumerate(losses_by_class): ax.text(i, value+0.04, f'{value:.3f}', ha='center')
ax.set_ylim(0, losses_by_class.max()*1.17)
plt.tight_layout(); plt.show()

## 8. ¿Cómo afecta la confianza a CE? (diapositivas 32–33)

Supongamos que **A = Puente es la etiqueta verdadera**:

- Modelo 1: $p=[0.55,0.40,0.05]$, predice A con 55%.
- Modelo 2: $p=[0.95,0.04,0.01]$, también predice A, pero con mayor confianza.
- Modelo 3: $p=[0.01,0.98,0.01]$, predice B con altísima confianza: está equivocado.

Los tres se comparan mediante **la probabilidad asignada a A**, no solamente con el indicador de acierto/error.

In [ ]:
modelos = {
    'Acierto poco seguro': np.array([.55,.40,.05]),
    'Acierto muy seguro': np.array([.95,.04,.01]),
    'Error muy seguro': np.array([.01,.98,.01]),
}
for nombre, probs in modelos.items():
    pred = np.argmax(probs)
    acc = float(pred == 0)
    ce = -np.log(probs[0])
    print(f'{nombre:23} -> predice {CLASES[pred]:9}; p(A)={probs[0]:.2f}; accuracy={acc:.0f}; CE={ce:.6f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for nombre, probs in modelos.items():
    axes[0].plot(CLASES, probs, marker='o', label=nombre)
axes[0].set_ylabel('Probabilidad asignada')
axes[0].set_ylim(0, 1)
axes[0].set_title('Las distribuciones predichas')
axes[0].legend(fontsize=8)
axes[1].bar(list(modelos), [-np.log(v[0]) for v in modelos.values()])
axes[1].tick_params(axis='x', labelrotation=15)
axes[1].set_ylabel('CE / NLL (nats)')
axes[1].set_title('Pérdida con A como clase verdadera')
plt.tight_layout(); plt.show()

## 9. Del ejemplo al lote: pérdida media, suma y accuracy (diapositivas 34–35 y 42–43)

Sea un lote de $N$ imágenes, con logits $z^{(i)}$, etiqueta correcta $y_i$ y probabilidades $p^{(i)}$.

- **Pérdida por observación:** $\ell_i=-\ln p^{(i)}_{y_i}$.
- **NLL total:** $\mathrm{NLL}_{\rm total}=\sum_i\ell_i$.
- **CE media del lote:** $\displaystyle\overline{\mathrm{CE}}=\frac1N\sum_i\ell_i$.
- **Clase predicha:** $\hat y_i=\arg\max_k p^{(i)}_k$.
- **Accuracy:** $\displaystyle\mathrm{Accuracy}=\frac1N\sum_{i=1}^N\mathbb1[\hat y_i=y_i]$.

**Interpretación:** accuracy indica cuántas etiquetas acertamos; CE penaliza las probabilidades asignadas a la clase verdadera. Dos modelos con la misma accuracy pueden tener CE diferente.

In [ ]:
# Cada fila es una fotografía; columnas = Puente, Edificio, Carretera.
Z_LOTE = np.array([[2.0,1.0,0.0],
                   [0.0,2.0,1.0],
                   [1.0,0.0,2.0],
                   [3.0,1.0,0.0],
                   [2.0,1.0,0.0]])
Y_LOTE = np.array([0,1,2,1,0])
P_LOTE = softmax_estable(Z_LOTE, axis=1)
P_CORRECTA = P_LOTE[np.arange(len(Y_LOTE)), Y_LOTE]
NLL_LOTE = -np.log(P_CORRECTA)
PREDICCIONES = P_LOTE.argmax(axis=1)
ACIERTOS = (PREDICCIONES == Y_LOTE)

print('i | clase real | clase predicha | p(real) | NLL_i | ¿acierto?')
for i in range(len(Y_LOTE)):
    print(f'{i+1} | {CLASES[Y_LOTE[i]]:10} | {CLASES[PREDICCIONES[i]]:14} | {P_CORRECTA[i]:.6f} | {NLL_LOTE[i]:.6f} | {"Sí" if ACIERTOS[i] else "No"}')
print(f'\nNLL total: {NLL_LOTE.sum():.6f}')
print(f'CE media:  {NLL_LOTE.mean():.6f}')
print(f'Accuracy:  {ACIERTOS.mean():.4f} = {100*ACIERTOS.mean():.1f}%')
assert np.allclose(P_LOTE.sum(axis=1), 1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(np.arange(1, len(NLL_LOTE)+1), NLL_LOTE)
ax.axhline(NLL_LOTE.mean(), linestyle='--', label=f'CE media = {NLL_LOTE.mean():.3f}')
ax.set_xlabel('Imagen i del lote')
ax.set_ylabel('Pérdida individual NLL_i')
ax.set_title('Cada ejemplo aporta una pérdida diferente')
ax.set_xticks(np.arange(1, len(NLL_LOTE)+1))
ax.legend()
plt.tight_layout(); plt.show()

### Comparación de dos modelos con la misma accuracy

Para simplificar emplearemos un conjunto con tres etiquetas verdaderas $[A,B,C]$. En ambos modelos las tres clases más probables serán las correctas, por lo que **accuracy = 100%**. Sin embargo, la CE será diferente porque las probabilidades correctas no son iguales.

In [ ]:
Y_TRES = np.array([0,1,2])
P_MODELO_1 = np.array([[.55,.40,.05],[.25,.55,.20],[.15,.30,.55]])
P_MODELO_2 = np.array([[.95,.04,.01],[.02,.95,.03],[.01,.04,.95]])
for nombre, probs in [('Modelo moderado', P_MODELO_1), ('Modelo confiado', P_MODELO_2)]:
    pred = probs.argmax(axis=1)
    acc = (pred == Y_TRES).mean()
    loss = -np.log(probs[np.arange(3), Y_TRES]).mean()
    print(f'{nombre:18}: accuracy={acc:.1%}, CE media={loss:.6f}')

## 10. Estabilidad numérica: softmax y log-sum-exp (diapositivas 36–38, 51–52)

El cálculo directo $e^{1000}$ puede producir `inf` en coma flotante. Softmax tiene una propiedad útil:

$$\operatorname{softmax}(z)=\operatorname{softmax}(z-m),\qquad m=\max_k z_k.$$

Por lo tanto, calculamos primero $z-m$, cuyos valores son cero o negativos. **Eso evita overflow en la exponencial**.

Para calcular NLL sin tomar $\ln(0)$ debido al subdesbordamiento de probabilidades muy pequeñas, utilizamos:

$$\boxed{\mathrm{NLL}(z,y)=\operatorname{LSE}(z)-z_y},\qquad
\operatorname{LSE}(z)=m+\ln\sum_k e^{z_k-m}.$$

**Derivación:** $-\ln(e^{z_y}/\sum_k e^{z_k})=-z_y+\ln\sum_k e^{z_k}$.

In [ ]:
def logsumexp_estable(z, axis=-1):
    z = np.asarray(z, dtype=float)
    m = np.max(z, axis=axis, keepdims=True)
    result = m + np.log(np.exp(z-m).sum(axis=axis, keepdims=True))
    return np.squeeze(result, axis=axis)

def nll_desde_logits(z, indice_verdadero):
    z = np.asarray(z, dtype=float)
    return float(logsumexp_estable(z) - z[indice_verdadero])

z_grande = np.array([1000.0,999.0,998.0])
with np.errstate(over='ignore', invalid='ignore'):
    e_directos = np.exp(z_grande)
    p_no_estable = e_directos/e_directos.sum()
print('Exponentes directos:', e_directos)
print('Softmax ingenuo:', p_no_estable)
print('Softmax estable:', softmax_estable(z_grande))
print('NLL estable para clase 0:', nll_desde_logits(z_grande, 0))
print('NLL con logits [2,1,0] :', nll_desde_logits([2,1,0], 0))
assert np.allclose(softmax_estable(z_grande), softmax_estable([2,1,0]))

### Verificación de NLL por dos caminos

Para logits moderados, ambos métodos deben dar el mismo número: (a) calcular softmax y después $-\ln p_y$; (b) calcular directamente $\operatorname{LSE}(z)-z_y$. El segundo es preferible en código de entrenamiento.

In [ ]:
for indice in range(3):
    via_probs = -np.log(softmax_estable(z_demo)[indice])
    via_logits = nll_desde_logits(z_demo, indice)
    print(f'Clase {CLASES[indice]:10}: vía p={via_probs:.9f} | vía logits={via_logits:.9f} | diferencia={abs(via_probs-via_logits):.2e}')
    assert np.isclose(via_probs, via_logits)

## 11. Objetivos suaves, entropía y divergencia KL (diapositivas 39–41)

Una etiqueta **one-hot** contiene todo el peso en una clase. Una **soft label** (etiqueta suave) distribuye pesos entre varias clases, por ejemplo $q=[0.7,0.2,0.1]$.

Para cualquier par de distribuciones positivas $q,p$ definimos:

$$\underbrace{H(q)}_{\text{entropía de la verdad}}=-\sum_k q_k\ln q_k$$
$$\underbrace{H(q,p)}_{\text{entropía cruzada}}=-\sum_k q_k\ln p_k$$
$$\underbrace{D_{\mathrm{KL}}(q\Vert p)}_{\text{divergencia}}=\sum_k q_k\ln\frac{q_k}{p_k}$$

Relación fundamental:

$$\boxed{H(q,p)=H(q)+D_{\mathrm{KL}}(q\Vert p)}.$$

Si $q$ es **one-hot**, $H(q)=0$ y **CE = KL = NLL** de la clase correcta. Si $q$ es suave, **CE y KL generalmente no tienen el mismo valor**, aunque comparten mínimo respecto a $p$ cuando $q$ permanece fija.

Convención matemática: $0\ln0=0$ y el sumando asociado a $q_k=0$ en KL es cero cuando $p_k>0$.

In [ ]:
def entropy(q):
    q = np.asarray(q, dtype=float)
    if np.any(q < 0) or not np.isclose(q.sum(), 1):
        raise ValueError('q debe ser una distribución')
    return float(-np.sum(q[q > 0] * np.log(q[q > 0])))

def kl_divergence(q, p):
    q = np.asarray(q, dtype=float); p = np.asarray(p, dtype=float)
    if q.shape != p.shape or np.any(p <= 0):
        raise ValueError('q y p deben tener igual forma y p > 0')
    mask = q > 0
    return float(np.sum(q[mask] * np.log(q[mask] / p[mask])))

q_one_hot = np.array([1.0, 0.0, 0.0])
q_soft = np.array([0.7, 0.2, 0.1])
for label, q in [('one-hot', q_one_hot), ('suave', q_soft)]:
    H = entropy(q)
    CE = cross_entropy(q, p_manual)
    KL = kl_divergence(q, p_manual)
    print(f'{label:8}: H(q)={H:.6f}; CE(q,p)={CE:.6f}; KL(q||p)={KL:.6f}; H+KL={H+KL:.6f}')
    assert np.isclose(CE, H+KL)
assert entropy(q_one_hot) == 0

### ¿Qué sucede si el modelo predice exactamente la distribución objetivo?

Si $p=q$ (para etiquetas suaves con componentes positivos), entonces $D_{KL}(q\Vert p)=0$, pero la entropía cruzada vale $H(q)$, que puede ser mayor que cero. Por ello **CE igual a cero no es requisito para una predicción perfecta con soft labels**.

In [ ]:
print('Objetivo suave:', q_soft)
print('Predicción exactamente igual al objetivo')
print('H(q)  =', entropy(q_soft))
print('CE    =', cross_entropy(q_soft,q_soft))
print('KL    =', kl_divergence(q_soft,q_soft))

## 12. Laboratorio experimental: variar logits y observar *todos* los indicadores

En la siguiente función evaluaremos para **una observación**:

- Logits originales y normalizados para el cálculo estable.
- Exponenciales, denominador y distribución softmax.
- Clase predicha por `argmax` y etiqueta verdadera one-hot.
- Probabilidad asignada a la clase verdadera.
- NLL desde softmax y desde log-sum-exp.
- CE, $H(q)$ y KL, además de accuracy individual (0 o 1).

En la etapa inicial usamos una etiqueta one-hot; más adelante puedes sustituirla por una etiqueta suave para explorar la relación CE = H + KL.

In [ ]:
def informe_completo(logits, etiqueta_verdadera, objetivo=None, mostrar=True):
    logits = np.asarray(logits, dtype=float)
    K = logits.size
    if logits.ndim != 1 or K != len(CLASES):
        raise ValueError('Se esperan exactamente tres logits')
    shifted = logits - logits.max()
    exponentes = np.exp(shifted)
    total = exponentes.sum()
    p = exponentes / total
    y = int(etiqueta_verdadera)
    q = one_hot(y,K) if objetivo is None else np.asarray(objetivo, dtype=float)
    pred = int(p.argmax())
    resultado = dict(
        logits=logits, logits_estables=shifted, exponentes=exponentes,
        denominador=total, probabilidades=p, suma_probabilidades=p.sum(),
        objetivo=q, clase_real=CLASES[y], clase_predicha=CLASES[pred],
        probabilidad_real=p[y], NLL_via_softmax=-np.log(p[y]),
        NLL_desde_logits=nll_desde_logits(logits,y), CE=cross_entropy(q,p),
        entropia=entropy(q), KL=kl_divergence(q,p), accuracy_individual=int(pred==y))
    if mostrar:
        for nombre, valor in resultado.items():
            print(f'{nombre:23}: {np.round(valor,6) if isinstance(valor,np.ndarray) else valor}')
        print('Consistencia CE = H(q) + KL:', np.isclose(resultado['CE'], resultado['entropia']+resultado['KL']))
    return resultado

informe = informe_completo([2,1,0], etiqueta_verdadera=0)

### Exploración visual de CE cuando cambia un logit

Mantenemos $z_B=1$, $z_C=0$ y la etiqueta verdadera **A**. Variamos solo $z_A$ entre $-2$ y $6$. Para cada cambio calculamos la probabilidad $p_A$ y la pérdida $-\ln p_A$. **Observa la relación inversa:** si $p_A$ aumenta, CE disminuye.

In [ ]:
valores_zA = np.linspace(-2,6,100)
prob_A = np.array([softmax_estable([zA,1,0])[0] for zA in valores_zA])
ce_A = -np.log(prob_A)
fig, axes = plt.subplots(1, 2, figsize=(11,3.7))
axes[0].plot(valores_zA, prob_A)
axes[0].set_xlabel('Logit z_A'); axes[0].set_ylabel('Probabilidad p_A')
axes[0].set_title('Al subir z_A, aumenta p_A')
axes[1].plot(valores_zA, ce_A)
axes[1].set_xlabel('Logit z_A'); axes[1].set_ylabel('CE = −ln(p_A)')
axes[1].set_title('Al subir p_A, disminuye CE')
plt.tight_layout(); plt.show()

## 13. Actividades para estudiantes (diapositivas 45–50)

**Indicaciones:** resuelve primero las operaciones manualmente. Luego escribe el código marcado como `TODO` y compáralo con las funciones de referencia. Los `TODO` están presentados como código válido que imprime instrucciones; reemplázalos por tus soluciones.

### Actividad A — Softmax y NLL con logaritmos exactos

Logits $z=[0,\ln 2,\ln 3]$. Calcula: (i) cada exponencial, (ii) denominador, (iii) $p$, (iv) NLL cuando la clase verdadera es **B**, y (v) qué clase predice argmax.

In [ ]:
# TODO A: reemplaza las líneas indicadas con tu solución.
z_ej_a = np.array([0.0, np.log(2), np.log(3)])
print('Pendiente: calcula exponentes, softmax, NLL para y=1 y argmax')
# exps_a = ...
# denominador_a = ...
# p_a = ...
# nll_a = ...
# print(...)

### Actividad B — La misma distribución, dos etiquetas distintas

Para $p=[0.7,0.2,0.1]$ calcula (i) CE si la clase real es A y (ii) CE si la clase real es C. Justifica por qué una es mayor, a pesar de ser la **misma predicción**.

In [ ]:
# TODO B: calcula CE con las etiquetas A y C usando la función cross_entropy.
p_ej_b = np.array([0.7,0.2,0.1])
print('Pendiente: comparar CE para q_A y q_C')
# ce_a = ...
# ce_c = ...
# print(...)

### Actividad C — Analiza un mini-batch

Se proporcionan tres imágenes y sus logits. Sus etiquetas verdaderas son $[A,B,C]$. Calcula por observación $p_y$, NLL y acierto; luego calcula CE media, NLL total y accuracy. ¿Cuál imagen aporta la mayor pérdida?

In [ ]:
# TODO C: utiliza softmax_estable sobre filas y selecciona p_y con indexación avanzada.
Z_EJ_C = np.array([[2.0,1.0,0.0],[3.0,1.0,0.0],[0.0,1.0,2.0]])
Y_EJ_C = np.array([0,1,2])
print('Pendiente: probabilidades, NLL por observación, media y accuracy')
# P_C = ...
# p_real_c = ...
# nll_por_ejemplo_c = ...
# acc_c = ...
# print(...)

### Actividad D — Invarianza y estabilidad

Comprueba numéricamente que $\mathrm{softmax}([2,1,0])$ es igual a $\mathrm{softmax}([1002,1001,1000])$, aunque `np.exp` directo no sea estable. Explica qué propiedad algebraica lo permite.

In [ ]:
# TODO D: verifica la igualdad con np.allclose y explica el resultado.
z_d1 = np.array([2.0,1.0,0.0])
z_d2 = np.array([1002.0,1001.0,1000.0])
print('Pendiente: comparación estable e interpretación')
# iguales = ...
# print(...)

### Actividad E — Entropía y divergencia KL

Sea $q=[0.7,0.2,0.1]$ y $p=[0.5,0.3,0.2]$. Calcula $H(q)$, CE y KL. Comprueba que $\mathrm{CE}=H(q)+D_{KL}(q\Vert p)$; interpreta qué representa cada número.

In [ ]:
# TODO E: completa usando entropy, cross_entropy y kl_divergence.
q_e = np.array([.7,.2,.1])
p_e = np.array([.5,.3,.2])
print('Pendiente: H(q), CE, KL y verificación de identidad')
# H_e = ...
# CE_e = ...
# KL_e = ...
# print(...)

## 14. Autoevaluación: preguntas que debes poder responder

1. ¿Por qué un logit no es una probabilidad?
2. ¿Qué representa cada término de $p_k=e^{z_k}/\sum_j e^{z_j}$?
3. ¿Por qué una variación de $z_B$ afecta también a $p_A$ y $p_C$?
4. ¿Por qué $q=[0,1,0]$ no es lo mismo que $p=[0.665,0.245,0.090]$?
5. ¿Cómo se deduce CE desde la probabilidad categórica?
6. ¿Por qué CE puede diferir aunque accuracy sea igual?
7. ¿Por qué es mejor calcular CE directamente desde logits para valores extremos?
8. ¿Cuándo CE, NLL y KL coinciden numéricamente?
9. ¿Qué cambia con un objetivo suave $q=[0.7,0.2,0.1]$?
10. ¿Por qué multietiqueta no se modela generalmente con una única softmax?

**Conclusión:** La cadena conceptual es: **entrada → capas → logits → softmax → distribución predicha → comparación con la etiqueta real → pérdida → futura optimización**.

**Referencia:** Simon J. D. Prince, *Understanding Deep Learning*, cap. 5; material docente complementario asociado a la presentación de 56 diapositivas. No es un notebook oficial del autor.